# H9b no-site: exact replay of the genuine Kaggle submission (CPU FP32)

**Author:** Francisco Angulo de Lafuente — ORCID 0009-0001-1634-7063.

**Accepted submission:** 58112557 — historical public Panoptic Quality **0.39**.
**True uploaded CSV SHA-256:** `24b3ea1c8e151c3053c69e61bc3d0a0a09bd7171fe82c95f7c109ce407efdc60`.

This notebook reproduces the final H9b stage from **the original 180 frozen no-site crop archives** using both original FP32 refiner checkpoints and the exact published source. P91 (13 November 2026) generated the true uploaded file exactly; independent P94 checked direct file equality and all 1,241 RLE masks. It uses no test ground-truth, no labels during inference, and no Kaggle leaderboard optimization.

**Scope warning:** The frozen no-site crop archives, `h6/components.pkl`, and detector-produced inputs must be generated from the official Kaggle images/training data or supplied from a verified private cache; they are **not included in the public repository**. This is a reproducible *last-stage* notebook, **not yet a clean-checkout end-to-end JPEG-to-upload solution**. P88 and the H3 raw-image notebook document earlier stages separately. This limitation must not be hidden in the final judging form.

In [ ]:
from __future__ import annotations

import csv
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np

REPO = Path.cwd().resolve()
SOURCE = REPO / "release" / "source"
ASSETS = Path(os.environ.get("SOLARFIL_ASSET_ROOT", str(REPO / "release"))).resolve()
OUTPUT = Path(os.environ.get("SOLARFIL_REPLAY_OUTPUT", str(REPO / "results" / "h9b_true_upload_cpu"))).resolve()
ACCEPTED_SHA256 = "24b3ea1c8e151c3053c69e61bc3d0a0a09bd7171fe82c95f7c109ce407efdc60"
REFINERS = [
    ("resnet34", "refiner_ALL.pt", "probs_resnet34.npz",
     "46ff938b3a2228ce2a1114eb8aa8d9191572e41385e2a3745269be329725db04"),
    ("efficientnet_v2_s", "refiner_ALL_efficientnet_v2_s.pt", "probs_efficientnet_v2_s.npz",
     "bf559d15b4111d5d4cabddd8e828b64806eaba553ca2ed8a78629e50485a632c"),
]

def file_sha256(path: Path) -> str:
    """Calculate the complete SHA-256 without loading large weights into memory."""
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Repository:", REPO)
print("External frozen inputs:", ASSETS)
print("CPU-only output:", OUTPUT)

## 1. Preflight — verify frozen data and published checkpoint integrity

On the authorized IA-Server, the original staged H9b no-site assets are under
`/srv/almacen/nas-2tb-1/solar-filament-2026/final-release-staging-20261112`.
For a different Linux host, set `SOLARFIL_ASSET_ROOT` to the equivalent **privately reconstructed**
input directory. Do not redistribute competition images or training-label caches without permission.

**Explicit failure is intentional:** a missing archive cannot be silently substituted by an uploaded answer CSV.

In [ ]:
source_files = [
    SOURCE / "h9_infer.py", SOURCE / "h9_train.py",
    SOURCE / "h9_official.py", SOURCE / "h9_crops.py",
    SOURCE / "h9_score.py", SOURCE / "h6_features.py",
    SOURCE / "h6_official.py", SOURCE / "h6_rule.py",
    SOURCE / "score_maps.py",
]
asset_files = [
    ASSETS / "dets_official.pkl",
    ASSETS / "h6" / "components.pkl",
    ASSETS / "submission.csv",
    ASSETS / "h9" / "official" / "crops_report.json",
] + [ASSETS / "models" / entry[1] for entry in REFINERS]
missing = [str(path) for path in source_files + asset_files if not path.is_file()]
crops = ASSETS / "h9" / "official" / "crops"
if not crops.is_dir():
    missing.append(str(crops))
if missing:
    raise FileNotFoundError(
        "Cannot claim full replay: required public source or PRIVATE frozen inputs missing:\n"
        + "\n".join(missing)
    )
archives = sorted(crops.glob("*.npz"))
assert len(archives) == 180, f"Expected 180 official unlabeled test-image crop archives; got {len(archives)}"
report = json.loads((ASSETS / "h9" / "official" / "crops_report.json").read_text())
assert report["no_site"] is True and len(report["per_image"]) == 180
for encoder, name, out_name, expected_sha in REFINERS:
    observed = file_sha256(ASSETS / "models" / name)
    assert observed == expected_sha, f"Checkpoint hash mismatch: {encoder} {observed}"
print("PREFLIGHT_PASS: 180 no-site archives, 2 original model hashes, all source inputs")

## 2. Re-run both original refinement networks on CPU in FP32

This reproduces the P91 execution algorithm rather than reusing its output:
original four-flip test-time augmentation, the same quantization rule, the
original trained model classes and deterministic crop ordering. CPU is chosen
because earlier GPU and CPU replays were **not pixel-exact**. No CUDA process
is allowed in this stage.

The results are checkpointed by model. Existing complete outputs are reused
only if they contain all 180 expected image keys; the final full-file SHA check
is mandatory regardless.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({
    "CUDA_VISIBLE_DEVICES": "",
    "OMP_NUM_THREADS": "2",
    "MKL_NUM_THREADS": "2",
    "OPENBLAS_NUM_THREADS": "2",
    "SOLARFIL_ASSET_ROOT": str(ASSETS),
    "PYTHONPATH": os.pathsep.join([str(SOURCE), str(REPO), environment.get("PYTHONPATH", "")]),
})
all_stems = {path.stem for path in archives}
for encoder, filename, probability_filename, expected_sha in REFINERS:
    target = OUTPUT / probability_filename
    if target.exists():
        with np.load(target, allow_pickle=False) as existing:
            assert set(existing.files) == all_stems, f"Cannot resume incomplete {target.name}"
    else:
        subprocess.run([
            sys.executable, str(SOURCE / "h9_infer.py"),
            "--weights", str(ASSETS / "models" / filename),
            "--encoder", encoder,
            "--crops", str(crops),
            "--out", str(target),
            "--device", "cpu",
        ], env=environment, check=True, cwd=str(REPO))
        with np.load(target, allow_pickle=False) as created:
            assert set(created.files) == all_stems
    print(encoder, "verified output:", target.name, file_sha256(target))

## 3. Reconstruct instances and the actual uploaded CSV

The code below runs the published native `h9_official.py paste` function.
Unlike the earlier stage-two replay, the **correct reference** is the genuine
uploaded file hash `24b3ea1c...`, not the misleadingly named intermediate
`submission_58112557.csv` (SHA-256 `67f02e4d...`).

In [ ]:
crop_link = OUTPUT / "crops"
if not crop_link.is_symlink():
    if crop_link.exists():
        raise RuntimeError("Existing output/crops must be a verified symlink; refusing ambiguous input")
    crop_link.symlink_to(crops, target_is_directory=True)
assert crop_link.resolve() == crops.resolve(), "Crop archive path changed"
csv_file = OUTPUT / "submission_h9b.csv"
if not csv_file.exists():
    subprocess.run([
        sys.executable, str(SOURCE / "h9_official.py"), "paste",
        "--out", str(OUTPUT),
        "--probs", *(str(OUTPUT / entry[2]) for entry in REFINERS),
    ], env=environment, check=True, cwd=str(REPO))
if not csv_file.is_file():
    raise RuntimeError("Native H9b paste produced no CSV")
with csv_file.open("r", encoding="utf-8", newline="") as stream:
    reader = csv.DictReader(stream)
    assert reader.fieldnames == ["filament_id", "segmentation_rle"]
    rows = list(reader)
ids = [row["filament_id"] for row in rows]
assert len(ids) == len(set(ids)) == 1241
assert len({name.rsplit("_", 1)[0] for name in ids}) == 180
observed_sha = file_sha256(csv_file)
assert observed_sha == ACCEPTED_SHA256, (
    "Genuine upload NOT reproduced. Do not report parity. "
    f"Observed={observed_sha}, accepted={ACCEPTED_SHA256}"
)
print("TRUE_UPLOAD_EXACT_REPLAY_PASS: 1241 rows, 180 images, SHA-256", observed_sha)

## 4. Integrity receipt, scope and scientific limitations

Write a machine-readable receipt only after all checks pass. The historical P91
and independent P94 audits succeeded with the true-upload hash on 13 November 2026.
This notebook makes no statement about PQ on test labels, unseen leaderboard
performance, or raw-JPEG-to-crop reproduction.

The public [P94 provenance correction](../docs/submission/P94_TRUE_UPLOAD_EXACT_PARITY_RESULT_20261113.md)
and [independent P94 JSON receipt](../docs/submission/P94_TRUE_UPLOAD_EXACT_PARITY_RECEIPT_20261113.json)
record the actual verified result. An organizer still needs the clean-input
preprocessing stages, which are under separate H3/P88 audits; the mandatory
final Google Form remains a separate action, not performed by a notebook.

In [ ]:
receipt = {
    "experiment": "H9B_TRUE_UPLOAD_CPU_FP32_NOTEBOOK",
    "verified_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "accepted_submission_id": 58112557,
    "accepted_public_pq_historical": 0.39,
    "uploaded_csv_sha256": ACCEPTED_SHA256,
    "computed_csv_sha256": observed_sha,
    "csv_exactly_matches_real_upload": observed_sha == ACCEPTED_SHA256,
    "predicted_instance_rows": len(rows),
    "source_images": len({name.rsplit("_", 1)[0] for name in ids}),
    "no_site_crops_verified": len(archives),
    "original_refiner_checkpoints": {
        encoder: file_sha256(ASSETS / "models" / filename)
        for encoder, filename, _out_name, _sha in REFINERS
    },
    "refiner_probability_sha256": {
        encoder: file_sha256(OUTPUT / probability_filename)
        for encoder, _filename, probability_filename, _sha in REFINERS
    },
    "test_annotations_read": False,
    "hidden_leaderboard_used": False,
    "kaggle_submissions_made": 0,
    "cuda_processes_started": 0,
    "frozen_crops_required": True,
    "end_to_end_original_jpeg_to_crop_verified": False,
}
receipt_path = OUTPUT / "true_upload_reproduction_receipt.json"
if receipt_path.exists():
    assert json.loads(receipt_path.read_text())["computed_csv_sha256"] == observed_sha
else:
    receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(receipt, indent=2, sort_keys=True))